# Curriculum 01 · Lab 2 — Token-Aware Splitting

**Goal:** Run the *same* budget number (300) through two splitters where the
number means different units — tokens vs characters — and measure both outputs
in real token space with tiktoken.

```
Splitter A : TokenSplitter        → TokenTextSplitter (300 TOKENS)
Splitter B : DocumentProcessor    → RecursiveCharacterTextSplitter (300 CHARACTERS)
Corpus     : 3 public-domain Project Gutenberg novels (Data/corpus/gutenberg/)
Tokenizer  : tiktoken cl100k_base (the gpt-4 encoder)
```

The second decision in every RAG pipeline is: **what unit do I budget chunks
in?** A token budget is the unit the LLM actually bills against; a character
budget says nothing about tokens — a 300-character chunk can cost anywhere from
~20 to 300+ tokens depending on how token-dense the text is (markdown markup,
code and symbols tokenize far heavier than plain prose).

This lab measures both splitters' output in real token space and reports, per
splitter: chunk count, average / min / max tokens per chunk, and the spread
(population std dev). Compare against lab 01 (fixed vs recursive character
splitting) and lab 03 (markdown structure): those split on size and structure,
this one splits on tokens.

## 0 · Setup — environment & imports

**WHAT:** Installs the two packages this lab needs beyond the base repo
(`tiktoken` for real token counts, `langchain-text-splitters` for the
`RecursiveCharacterTextSplitter` behind `DocumentProcessor`), then imports the
repo's component library (`src/loaders/gutenberg.GutenbergLoader`,
`src/splitters/token_splitter.TokenSplitter`,
`src/splitters/recursive.DocumentProcessor`) and sets the run configuration.

**WHY:** The lab compares two splitters from the repo's swappable component
library — the same blocks you swap in any pipeline. The setup normalizes the
working directory to the repo root (a notebook kernel starts in the notebook's
own folder, unlike the `.py` lab which you run from the repo root) and puts it
on `sys.path`, so the repo-root component library imports and the `Data/...`
corpus paths resolve exactly as in the `.py`.

**WHAT TO EXPECT:** No output from the import cell; the `%pip install` cell
prints a short "already satisfied" summary if the packages are present.

In [1]:
# Needed for THIS lab only:
#   tiktoken                 → real token counts (cl100k_base, the gpt-4 tokenizer)
#   langchain-text-splitters → RecursiveCharacterTextSplitter (behind DocumentProcessor)
%pip install tiktoken langchain-text-splitters


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import statistics
import sys
from pathlib import Path

import tiktoken
from langchain_core.documents import Document

# Make the repo-root component library importable and the repo-root-relative
# corpus paths (DOC_PATHS below) resolve exactly as when the .py lab runs from
# the repo root. The .py derives REPO_ROOT from __file__; a notebook kernel
# instead starts in the notebook's own directory, so walk up from cwd until we
# find the repo root (the folder holding Data/corpus/gutenberg/), chdir there,
# and put it on sys.path.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "Data" / "corpus" / "gutenberg").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError(
            "Could not locate repo root (missing Data/corpus/gutenberg/)"
        )
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from loaders.gutenberg import GutenbergLoader  # noqa: E402
from splitters.recursive import DocumentProcessor  # noqa: E402
from splitters.token_splitter import TokenSplitter  # noqa: E402

# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the comparison
# --------------------------------------------------------------------------
TOKEN_CHUNK_SIZE = 300            # budget in TOKENS — the unit the LLM bills
TOKEN_CHUNK_OVERLAP = 30
CHAR_CHUNK_SIZE = 300             # same number, different unit: CHARACTERS
CHAR_CHUNK_OVERLAP = 30
DOC_PATHS = [
    Path("Data/corpus/gutenberg/pride-and-prejudice.txt"),
    Path("Data/corpus/gutenberg/moby-dick.txt"),
    Path("Data/corpus/gutenberg/a-tale-of-two-cities.txt"),
]

# cl100k_base = the gpt-4 tokenizer; same encoder token_splitter.py measures
# with. Kept as a constant so the lab can be re-run under another encoding.
ENCODER_NAME = "cl100k_base"

## 1 · Load — GutenbergLoader strips the license preamble

`GutenbergLoader` reads each public-domain novel from `Data/corpus/gutenberg/`
and strips the Project Gutenberg license preamble/footer (everything outside
the `*** START … ***` / `*** END … ***` markers) before wrapping the text as a
`Document` tagged with its source path. That boilerplate is noise for chunking
and retrieval, so it never reaches the splitters.

The three books — Pride and Prejudice, Moby-Dick, A Tale of Two Cities — are
long prose documents (~0.7–1.2M chars each), which is exactly the regime where
a character budget and a token budget diverge.

In [3]:
# --------------------------------------------------------------------------
# 2. Load — GutenbergLoader strips the license preamble, wraps as Documents
# --------------------------------------------------------------------------
def load_documents(paths: list[Path]) -> list[Document]:
    """Load each Gutenberg book as a Document tagged with its source path."""
    docs: list[Document] = []
    for path in paths:
        docs.extend(GutenbergLoader(path, strip=True).load())
    return docs

In [4]:
enc = tiktoken.get_encoding(ENCODER_NAME)
print(f"Encoding: {ENCODER_NAME} (gpt-4 tokenizer)")
print(
    f"Budget: {TOKEN_CHUNK_SIZE} tokens vs {CHAR_CHUNK_SIZE} characters, "
    f"overlap {TOKEN_CHUNK_OVERLAP}\n"
)

docs = load_documents(DOC_PATHS)
print(f"Loaded {len(docs)} document(s): {[p.name for p in DOC_PATHS]}")
for doc in docs:
    print(f"  {Path(doc.metadata['source']).name}: {len(doc.page_content):,} chars")

Encoding: cl100k_base (gpt-4 tokenizer)
Budget: 300 tokens vs 300 characters, overlap 30

Loaded 3 document(s): ['pride-and-prejudice.txt', 'moby-dick.txt', 'a-tale-of-two-cities.txt']
  pride-and-prejudice.txt: 728,769 chars
  moby-dick.txt: 1,218,971 chars
  a-tale-of-two-cities.txt: 757,631 chars


## 2 · Split — TokenSplitter vs DocumentProcessor

The same budget number `300` goes through two splitters where the number means
different units:

* **`TokenSplitter`** wraps `TokenTextSplitter` and cuts on **token**
  boundaries, so every chunk is guaranteed to stay inside the token budget the
  LLM actually bills against.
* **`DocumentProcessor`** wraps `RecursiveCharacterTextSplitter` and cuts on
  **character** counts, which say nothing about tokens.

Both get the same three documents and the same overlap (30). The only
difference is the unit the budget is enforced in.

In [5]:
token_splitter = TokenSplitter(
    chunk_size=TOKEN_CHUNK_SIZE, chunk_overlap=TOKEN_CHUNK_OVERLAP
)
char_splitter = DocumentProcessor(
    chunk_size=CHAR_CHUNK_SIZE, chunk_overlap=CHAR_CHUNK_OVERLAP
)

token_chunks = token_splitter.split_docs(docs)
char_chunks = char_splitter.split_docs(docs)
print(
    f"TokenSplitter ({TOKEN_CHUNK_SIZE} tokens): {len(token_chunks)} chunk(s); "
    f"DocumentProcessor ({CHAR_CHUNK_SIZE} chars): {len(char_chunks)} chunk(s)\n"
)

TokenSplitter (300 tokens): 2690 chunk(s); DocumentProcessor (300 chars): 12023 chunk(s)



## 3 · Measure — token consumption in real token space

Chunk counts alone hide the story: the character splitter produced ~4.5× more
chunks for the same nominal budget. To see why, measure **every chunk in real
token space** with tiktoken (`cl100k_base` — the same encoder
`src/splitters/token_splitter.py` uses internally). The helpers below summarize a
chunk population: count, average / min / max tokens per chunk, and the spread
(population std dev).

In [6]:
# --------------------------------------------------------------------------
# 3. Token-count — measure every chunk in real token space
# --------------------------------------------------------------------------
def token_counts(chunks: list[Document], enc: tiktoken.Encoding) -> list[int]:
    """Measure each chunk's real token consumption (not its char count)."""
    return [len(enc.encode(chunk.page_content)) for chunk in chunks]


def chunk_stats(counts: list[int]) -> dict[str, float]:
    """Summarize a chunk population in token space: count/avg/min/max/std."""
    n = len(counts)
    return {
        "chunks": n,
        "avg": sum(counts) / n,
        "min": min(counts),
        "max": max(counts),
        "std": statistics.pstdev(counts),  # full population of chunks
    }


def preview(text: str, limit: int = 200) -> str:
    """Truncate a chunk's content for printing."""
    return text[:limit] + ("..." if len(text) > limit else "")


def print_stats_row(label: str, counts: list[int], note: str) -> None:
    """Print one row of the comparison table, all numbers in token space."""
    s = chunk_stats(counts)
    print(
        f"{label:<38} {s['chunks']:>6} {s['avg']:>7.1f} "
        f"{s['min']:>4.0f} {s['max']:>4.0f} {s['std']:>7.1f}   {note}"
    )

In [7]:
token_tokens = token_counts(token_chunks, enc)
char_tokens = token_counts(char_chunks, enc)

# Same nominal budget "300" — the only difference is the unit it is
# enforced in. All numbers below are tiktoken token counts.
print("Chunk count and token consumption, measured with tiktoken:")
print(f"{'splitter':<38} {'chunks':>6} {'avg':>7} {'min':>4} {'max':>4} {'std dev':>7}")
print("-" * 88)
print_stats_row(
    f"TokenSplitter ({TOKEN_CHUNK_SIZE} tokens)",
    token_tokens,
    "bounded: every chunk <= budget",
)
print_stats_row(
    f"DocumentProcessor ({CHAR_CHUNK_SIZE} chars)",
    char_tokens,
    "no token ceiling: wide spread",
)
print("-" * 88)
spread = max(char_tokens) / min(char_tokens)
print(
    f"Token chunks: {min(token_tokens)}-{max(token_tokens)} tokens — uniform, "
    f"bounded at the {TOKEN_CHUNK_SIZE}-token budget (the few tokens over "
    f"300 are each document's final-chunk remainder)."
)
print(
    f"Char chunks: {min(char_tokens)}-{max(char_tokens)} tokens — a "
    f"{spread:.1f}x spread for the same '300' budget."
)

Chunk count and token consumption, measured with tiktoken:
splitter                               chunks     avg  min  max std dev
----------------------------------------------------------------------------------------
TokenSplitter (300 tokens)               2690   275.0   45  305    26.5   bounded: every chunk <= budget
DocumentProcessor (300 chars)           12023    55.0    1  103    19.7   no token ceiling: wide spread
----------------------------------------------------------------------------------------
Token chunks: 45-305 tokens — uniform, bounded at the 300-token budget (the few tokens over 300 are each document's final-chunk remainder).
Char chunks: 1-103 tokens — a 103.0x spread for the same '300' budget.


## 4 · Escalation — the repo default char budget, in token terms

`DocumentProcessor`'s default is `chunk_size=1000` **characters**. Raise the
character budget and watch the token cost climb: nothing in the config mentions
tokens, yet the chunks consume most of the 300-token budget.

In [8]:
default_char_splitter = DocumentProcessor()  # repo default: 1000 chars
default_char_chunks = default_char_splitter.split_docs(docs)
default_char_tokens = token_counts(default_char_chunks, enc)
s = chunk_stats(default_char_tokens)
print("\nEscalation — DocumentProcessor at its default budget (1000 chars):")
print(
    f"  {s['chunks']:.0f} chunk(s), avg {s['avg']:.1f}, "
    f"min {s['min']:.0f}, max {s['max']:.0f} tokens per chunk"
)
print(
    f"  -> a '1000-character' budget silently consumed up to "
    f"{s['max']:.0f} tokens per chunk, ~{100 * s['max'] / TOKEN_CHUNK_SIZE:.0f}% "
    f"of the {TOKEN_CHUNK_SIZE}-token budget."
)


Escalation — DocumentProcessor at its default budget (1000 chars):
  3825 chunk(s), avg 186.7, min 6, max 319 tokens per chunk
  -> a '1000-character' budget silently consumed up to 319 tokens per chunk, ~106% of the 300-token budget.


## 5 · Side by side — one real chunk from each splitter

The first chunk of each splitter, printed with its measured token count and
character count. Both start at the same book — the difference is where the
splitter chose to cut.

In [9]:
print("\nSide by side — first chunk of each splitter:")
for label, chunk, tokens in [
    ("TokenSplitter", token_chunks[0], token_tokens[0]),
    ("DocumentProcessor", char_chunks[0], char_tokens[0]),
]:
    print(f"  {label}: {tokens} tokens, {len(chunk.page_content)} chars")
    print(f"    {preview(chunk.page_content)!r}")


Side by side — first chunk of each splitter:
  TokenSplitter: 60 tokens, 407 chars
    'PRIDE AND PREJUDICE ***\n\n\n\n\n                            [Illustration:\n\n                             GEORGE ALLEN\n                               PUBLISHER\n\n                        156 CHARING CROSS RO...'
  DocumentProcessor: 38 tokens, 241 chars
    'PRIDE AND PREJUDICE ***\n\n\n\n\n                            [Illustration:\n\n                             GEORGE ALLEN\n                               PUBLISHER\n\n                        156 CHARING CROSS RO...'


## 6 · Takeaway — token budget vs character budget

The same nominal `300` produced wildly different chunk populations depending on
the unit it was enforced in. A token-aware splitter is the only one that can
guarantee your chunks fit the context window you actually pay for.

In [10]:
print("\nTakeaway: a character budget is not a token budget.")
print(
    f"- TokenSplitter({TOKEN_CHUNK_SIZE} tokens): every chunk is bounded at "
    f"the {TOKEN_CHUNK_SIZE}-token budget (measured "
    f"{min(token_tokens)}-{max(token_tokens)}); the budget is enforced in "
    f"the unit the LLM bills."
)
print(
    f"- DocumentProcessor({CHAR_CHUNK_SIZE} chars): the same '300' produced "
    f"chunks of {min(char_tokens)}-{max(char_tokens)} tokens ({spread:.1f}x spread). "
    f"Character count never determines token count — markdown, code and "
    f"symbols tokenize far heavier than prose — so no character budget "
    f"sets a token ceiling."
)
print(
    f"  (On these three novels the same nominal '300' produced "
    f"{len(char_tokens):,} char-budgeted chunks of "
    f"{min(char_tokens)}-{max(char_tokens)} tokens — a {spread:.0f}x spread, "
    f"driven by short fragments (chapter headings, whitespace runs) at the "
    f"low end. Plain prose runs ~5 chars/token, so a 300-char chunk lands "
    f"around {sum(char_tokens) // len(char_tokens)} tokens on average — well "
    f"under the 300-token budget — yet the character budget still sets no "
    f"token ceiling: at the repo's default 1000-char budget the max reached "
    f"{s['max']:.0f} tokens, ~{100 * s['max'] / TOKEN_CHUNK_SIZE:.0f}% of the "
    f"{TOKEN_CHUNK_SIZE}-token budget.)"
)
print(
    "- A token-aware splitter is the only one that can guarantee your "
    "chunks fit the context window you actually pay for."
)


Takeaway: a character budget is not a token budget.
- TokenSplitter(300 tokens): every chunk is bounded at the 300-token budget (measured 45-305); the budget is enforced in the unit the LLM bills.
- DocumentProcessor(300 chars): the same '300' produced chunks of 1-103 tokens (103.0x spread). Character count never determines token count — markdown, code and symbols tokenize far heavier than prose — so no character budget sets a token ceiling.
  (On these three novels the same nominal '300' produced 12,023 char-budgeted chunks of 1-103 tokens — a 103x spread, driven by short fragments (chapter headings, whitespace runs) at the low end. Plain prose runs ~5 chars/token, so a 300-char chunk lands around 54 tokens on average — well under the 300-token budget — yet the character budget still sets no token ceiling: at the repo's default 1000-char budget the max reached 319 tokens, ~106% of the 300-token budget.)
- A token-aware splitter is the only one that can guarantee your chunks fit the c

## What you should notice

* **A token budget is the unit the LLM bills.** `TokenSplitter` chunks came out
  uniform and bounded (45–305 tokens for a 300-token budget — the few tokens
  over 300 are each document's final-chunk remainder). The budget is enforced
  in the unit the model actually pays for.
* **The same "300" as characters is a different budget.** `DocumentProcessor`
  produced 12,023 chunks of 1–103 tokens — a 103× spread for the same nominal
  number. Character count never determines token count.
* **Character budgets set no token ceiling.** At the repo's default 1000-char
  budget, chunks reached 319 tokens — ~106% of the 300-token budget — with
  nothing in the config mentioning tokens.
* **Token-dense text is the trap.** Markdown markup, code and symbols tokenize
  far heavier than plain prose, so a character budget that looks safe on prose
  silently overflows on structured text.

## Exercises

1. **Change the budget.** Rerun with `TOKEN_CHUNK_SIZE = 500` and
   `CHAR_CHUNK_SIZE = 500`. How do the chunk counts and spreads change? Does
   the character splitter's spread grow or shrink?
2. **Tune the overlap.** Set `TOKEN_CHUNK_OVERLAP = 0` and compare the token
   distribution — what does overlap cost in tokens?
3. **Try a different encoder.** Set `ENCODER_NAME = "o200k_base"` (the gpt-4o
   tokenizer) and re-measure. Do the token counts move?
4. **Compare with lab 01.** Lab 01 compared fixed vs recursive character
   splitting; this lab compares token vs character budgets. Which splitter
   would you pick for a code-heavy corpus, and why?